## Łączenie wszytkich plików .nc

In [2]:
import xarray as xr
import glob

sciezka = "./dataset/era5/krakow_temp_*.nc" 
pliki = sorted(glob.glob(sciezka))

ds = xr.open_mfdataset(pliki, combine='by_coords', chunks={'valid_time': 1000})

ds.to_netcdf('./dataset/krakow_era5_2000_2025.nc')

print(ds)

<xarray.Dataset> Size: 27MB
Dimensions:     (valid_time: 227928, latitude: 4, longitude: 6)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2MB 2000-01-01 ... 2025-12-31T23:...
  * latitude    (latitude) float64 32B 50.2 50.1 50.0 49.9
  * longitude   (longitude) float64 48B 19.7 19.8 19.9 20.0 20.1 20.2
    number      int64 8B 0
    expver      (valid_time) <U4 4MB dask.array<chunksize=(744,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 22MB dask.array<chunksize=(744, 4, 6), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-11T14:34 GRIB to CDM+CF via cfgrib-0.9.1...


## Wyodrębnienie danych z pliku NetCDF

In [15]:
print("Wczytywanie ERA5...")
ds = xr.open_dataset('./dataset/krakow_era5_2000_2025.nc')

# Wybieramy najbliższy punkt dla Balic i Obserwatorium
# Balice: 50.077, 19.784 | Obserwatorium: 50.067, 19.959
era_balice = ds['t2m'].sel(latitude=50.077, longitude=19.784, method='nearest').to_dataframe().reset_index()
era_obs = ds['t2m'].sel(latitude=50.067, longitude=19.959, method='nearest').to_dataframe().reset_index()

# Konwersja K -> C i ujednolicenie nazw kolumn czasu
for df in [era_balice, era_obs]:
    df['temp_era5'] = df['t2m'] - 273.15
    # W ERA5 kolumna czasu często nazywa się 'time' lub 'valid_time'
    # Zmieniamy ją na 'timestamp', żeby łatwo łączyć
    if 'time' in df.columns:
        df.rename(columns={'time': 'timestamp'}, inplace=True)
    elif 'valid_time' in df.columns:
        df.rename(columns={'valid_time': 'timestamp'}, inplace=True)

Wczytywanie ERA5...


## Łączenie wszystkich plików dla stacji Kraków-Balice

In [3]:
import glob
import pandas as pd

# Scalanie wszystkich plików Balic
path = "dataset/imgw/balice/KRAKOW_BALICE_*.csv"
all_files = sorted(glob.glob(path))
df_total = pd.concat([pd.read_csv(f) for f in all_files])

# Opcjonalnie: stwórz kolumnę czasową dla wygody
df_total['dt'] = pd.to_datetime(df_total[['year', 'month', 'day', 'hour']])
df_total.to_csv("./dataset/krakow_balice_imgw_2000_2025.csv", index=False)

## Ujednolicenie danych IMGW do UTC

In [16]:
def prepare_imgw(path):
    print(f"Przetwarzanie {path}...")
    df = pd.read_csv(path)
    
    # Tworzymy czas lokalny z kolumn rok, mc, dz, gg
    df['dt_local'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    
    # Konwersja Lokalny -> UTC (naprawia błąd AmbiguousTimeError)
    df['timestamp'] = df['dt_local'].dt.tz_localize(
        'Europe/Warsaw', ambiguous='NaT', nonexistent='shift_forward'
    ).dt.tz_convert('UTC').dt.tz_localize(None)
    
    return df.dropna(subset=['timestamp'])[['timestamp', 'temp']]

imgw_balice = prepare_imgw('./dataset/krakow_balice_imgw_2000_2025.csv')
imgw_obs = prepare_imgw('./dataset/krakow_obserwatorium_imgw_2000_2025.csv')

Przetwarzanie ./dataset/krakow_balice_imgw_2000_2025.csv...
Przetwarzanie ./dataset/krakow_obserwatorium_imgw_2000_2025.csv...


## Łączenie danych ERA5 oraz IMGW

In [18]:
# Łączymy po kolumnie 'timestamp'
final_balice = pd.merge(era_balice[['timestamp', 'temp_era5']], imgw_balice, on='timestamp', how='inner')
final_obs = pd.merge(era_obs[['timestamp', 'temp_era5']], imgw_obs, on='timestamp', how='inner')

# Zapisujemy do plików
final_balice.to_csv('./dataset/final_balice.csv', index=False)
final_obs.to_csv('./dataset/final_obserwatorium.csv', index=False)

print("\n✅ Sukces! Pliki zapisane:")
print(f"- Balice: {len(final_balice)} wierszy")
print(f"- Obserwatorium: {len(final_obs)} wierszy")


✅ Sukces! Pliki zapisane:
- Balice: 227901 wierszy
- Obserwatorium: 28305 wierszy
